In [1]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time

@njit(fastmath=True, nogil=True)
def harvesting_kernel_final(u_c, k_static, steps, n_bins):
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    last_bin = 0
    warmup = 2000000 
    
    for i in range(steps + warmup):
        # 物理真理冷却公式
        ln_term = np.log(i + 100)
        mu_eff = u_c + k_static / (ln_term**2)
        x = 1 - mu_eff * x**2
        
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
            
        if i > warmup:
            current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
            if 0 <= current_bin < n_bins:
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
    return counts

def sniper_worker_final(task):
    seg_idx, start_n, end_n, k_opt = task
    U_C = 1.543689012
    N_BINS = 20000    
    STEPS = 10**8     
    SAVE_DIR = "riemann_10k_harvest_FINAL"
    
    try:
        t0 = time.time()
        counts = harvesting_kernel_final(U_C, k_opt, STEPS, N_BINS)
        P = csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        calc_k = (end_n - start_n) * 2 + 100 
        v0 = np.ones(N_BINS)
        vals, _ = eigs(P, k=calc_k, which='LM', tol=1e-4, v0=v0)
        
        # 纯正能量态提取 (打破对称性)
        pure_vals = vals[(np.abs(vals) > 0.4) & (vals.imag > 1e-7)]
        phases = np.sort(np.angle(pure_vals))
        
        if not os.path.exists(SAVE_DIR):
            os.makedirs(SAVE_DIR, exist_ok=True)
        np.save(os.path.join(SAVE_DIR, f"seg_{seg_idx}_n_{start_n}.npy"), phases)
        return f"DONE | Seg {seg_idx:3} | k={k_opt:.4f} | {time.time()-t0:.1f}s"
    except Exception as e:
        return f"FAIL | Seg {seg_idx:3} | {str(e)}"

if __name__ == "__main__":
    SAVE_DIR = "riemann_10k_harvest_FINAL"
    if not os.path.exists(SAVE_DIR): os.makedirs(SAVE_DIR)
    print(f"🚀 启动终极复现 (k0=4.7, beta=10.13)")
    
    K0, BETA = 4.7000, 10.13
    tasks = []
    for i in range(0, 10000, 100):
        mid_n = i + 50 if i + 50 > 1 else 2
        k_opt = K0 + BETA / np.log(mid_n) # 严格按照推导公式
        tasks.append((i//100, i, i+100, k_opt))

    with mp.Pool(processes=100) as pool:
        for res in pool.map(sniper_worker_final, tasks): print(f"  {res}")
    print(f"\n✅ 终极数据收割完毕！")

🚀 启动终极复现 (k0=4.7, beta=10.13)
  DONE | Seg   0 | k=7.2895 | 665.0s
  DONE | Seg   1 | k=6.7217 | 771.2s
  DONE | Seg   2 | k=6.5347 | 772.4s
  DONE | Seg   3 | k=6.4293 | 695.8s
  DONE | Seg   4 | k=6.3581 | 722.2s
  DONE | Seg   5 | k=6.3054 | 772.2s
  DONE | Seg   6 | k=6.2640 | 704.2s
  DONE | Seg   7 | k=6.2302 | 741.8s
  DONE | Seg   8 | k=6.2018 | 706.9s
  DONE | Seg   9 | k=6.1774 | 767.1s
  DONE | Seg  10 | k=6.1562 | 768.6s
  DONE | Seg  11 | k=6.1374 | 764.0s
  DONE | Seg  12 | k=6.1206 | 760.7s
  DONE | Seg  13 | k=6.1054 | 619.7s
  DONE | Seg  14 | k=6.0916 | 645.2s
  DONE | Seg  15 | k=6.0790 | 749.3s
  DONE | Seg  16 | k=6.0673 | 755.7s
  DONE | Seg  17 | k=6.0566 | 730.0s
  DONE | Seg  18 | k=6.0465 | 747.8s
  DONE | Seg  19 | k=6.0372 | 698.8s
  DONE | Seg  20 | k=6.0284 | 722.2s
  DONE | Seg  21 | k=6.0202 | 712.1s
  DONE | Seg  22 | k=6.0124 | 761.7s
  DONE | Seg  23 | k=6.0050 | 654.8s
  DONE | Seg  24 | k=5.9981 | 701.6s
  DONE | Seg  25 | k=5.9915 | 756.6s
  DONE |